In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / "src" / "utils").is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError("Project root with src/utils was not found")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from utils import (
    DEFAULT_CUBE_SIZES,
    BereaPatchDataset,
    DigitalCorePipeline,
    TOPOLOGY_FEATURE_DIM,
    TopologyAdaptiveRoutedUNet3D,
    check_required_dependencies,
)

check_required_dependencies()
device = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = ROOT / "outputs" / "networks"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT, "device:", device)


ROOT: f:\PycharmProjects\micro_ct device: cuda


In [2]:
CUBE_SIZES = list(DEFAULT_CUBE_SIZES)
SPLIT = "train"
MAX_SAMPLES = 20
USE_TARGET_MASK = True
THRESHOLD = 0.5
VOXEL_SIZE_M = 2.25e-6
SEG_CHECKPOINT = ROOT / "models" / "segmentation_best.pth"


In [3]:
dataset = BereaPatchDataset(ROOT, split=SPLIT, cube_size=CUBE_SIZES, use_raw_gray=False, noise_types=["none"], balance=True)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

seg_model = None
if not USE_TARGET_MASK:
    seg_model = TopologyAdaptiveRoutedUNet3D(base_channels=16, ctx_dim=64, ph_dim=TOPOLOGY_FEATURE_DIM, topology_dim=TOPOLOGY_FEATURE_DIM).to(device)
    checkpoint = torch.load(SEG_CHECKPOINT, map_location=device)
    seg_model.load_state_dict(checkpoint["model_state_dict"])

pipeline = DigitalCorePipeline(
    segmentation_model=seg_model,
    device=device,
    threshold=THRESHOLD,
    voxel_size=VOXEL_SIZE_M,
    mu=1.0e-3,
)
print("cube sizes:", CUBE_SIZES)
print("samples:", len(dataset))
display(dataset.df.groupby(["rock", "cube_size", "split"]).size().rename("samples").reset_index())


cube sizes: [64, 128, 192]
samples: 19662


,rock,cube_size,split,samples
0,BanderaBrown,64,train,3277
1,BanderaBrown,128,train,410
2,BanderaBrown,192,train,173
3,Berea,64,train,3277
4,Berea,128,train,410
5,Berea,192,train,173


In [ ]:
rows = []

for sample_id, batch in enumerate(tqdm(loader, desc="extract networks")):
    if MAX_SAMPLES is not None and sample_id >= MAX_SAMPLES:
        break

    if USE_TARGET_MASK:
        cube = batch["y"][0, 0].numpy() > 0.5
        input_is_pore_mask = True
    else:
        cube = batch["x"][0].numpy()
        input_is_pore_mask = False

    cube_size = int(batch["cube_size"][0])
    domain_size = (cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M, cube_size * VOXEL_SIZE_M)
    rock = batch["rock"][0]

    try:
        result = pipeline.run_cube(
            cube,
            input_is_pore_mask=input_is_pore_mask,
            domain_size=domain_size,
            include_ph=True,
            compute_openpnm_baseline=True,
        )
    except Exception as exc:
        print(f"sample {sample_id} skipped: {exc}")
        continue

    coord = batch["coord"][0].tolist()
    result.network.metadata.update({
        "sample_id": sample_id,
        "coord": coord,
        "rock": rock,
        "cube_size": cube_size,
        "openpnm_k": result.permeability.k_openpnm,
    })
    out_path = OUT_DIR / f"network_{SPLIT}_{sample_id:04d}.pt"
    torch.save(result.network, out_path)

    row = {
        "sample_id": sample_id,
        "path": str(out_path),
        "rock": rock,
        "cube_size": cube_size,
        "z": coord[0],
        "y": coord[1],
        "x": coord[2],
        **result.network.metadata,
    }
    if result.permeability.k_openpnm is not None:
        row.update({f"openpnm_{k}": v for k, v in result.permeability.k_openpnm.items()})
    rows.append(row)

summary = pd.DataFrame(rows)
summary_path = OUT_DIR / f"network_{SPLIT}_summary.csv"
summary.to_csv(summary_path, index=False)
summary


extract networks:   0%|          | 3/19662 [00:41<67:18:05, 12.32s/it] 